<a href="https://colab.research.google.com/github/monteiro-sara/algorithmic_trading/blob/main/German_Lobbyregister_Live.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Who lobbies Germany? — fixed live-data version

This notebook analyzes **current official open data** from the German Bundestag Lobbyregister.

The previous version downloaded records but assumed an outdated JSON nesting, so analytical fields became empty. This version instead:

- uses the current public `https://www.lobbyregister.bundestag.de/sucheJson` feed;
- inspects the live schema before parsing;
- supports both flat and nested response structures;
- validates that expenditure / employee / policy fields were really extracted;
- produces descriptive outputs even if an optional field is absent;
- never converts missing financial disclosure to €0.

Expenditure is reported in bands; descriptive analyses use the **band midpoint**.

In [1]:
import json, math, warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
PLOTLY_TEMPLATE = "plotly_white"

print("Ready.")

Ready.


## 1 — Pull the current Bundestag open-data feed

The request is made directly to the Bundestag-hosted JSON endpoint, with no local input dataset.

In [2]:
SOURCE_URL = "https://www.lobbyregister.bundestag.de/sucheJson"
PARAMS = {"sort": "FINANCIALEXPENSES_DESC"}
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; open-science-lobbyregister-analysis/2.0)",
    "Accept": "application/json",
}

r = requests.get(SOURCE_URL, params=PARAMS, headers=HEADERS, timeout=180)
r.raise_for_status()
payload = r.json()

print("Live Bundestag download successful")
print("HTTP:", r.status_code)
print("Request URL:", r.url)
print("Source:", payload.get("source", "Deutscher Bundestag Lobbyregister"))
print("Search timestamp:", payload.get("searchDate", "not supplied"))
print("Reported result count:", payload.get("resultCount", "not supplied"))
print("Top-level keys:", list(payload.keys()))

✅ Live Bundestag download successful
HTTP: 200
Request URL: https://www.lobbyregister.bundestag.de/sucheJson?sort=FINANCIALEXPENSES_DESC
Source: Deutscher Bundestag, Lobbyregister für die Interessenvertretung gegenüber dem Deutschen Bundestag und der Bundesregierung
Search timestamp: not supplied
Reported result count: 6941
Top-level keys: ['$schema', 'source', 'sourceUrl', 'sourceDate', 'jsonDocumentationUrl', 'searchUrl', 'searchParameters', 'resultCount', 'results']


## 2 — Inspect and normalize the live JSON

The parser first uses known direct paths and then falls back to recursive key discovery. This prevents a harmless nesting change from silently turning the entire analysis into `NaN`.

In [3]:
def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def get_path(obj, path):
    cur = obj
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return None
        cur = cur[key]
    return cur

def find_first_key(obj, key):
    if isinstance(obj, dict):
        if key in obj and obj[key] is not None:
            return obj[key]
        for v in obj.values():
            out = find_first_key(v, key)
            if out is not None:
                return out
    elif isinstance(obj, list):
        for v in obj:
            out = find_first_key(v, key)
            if out is not None:
                return out
    return None

def preferred(record, paths, key=None, default=None):
    for path in paths:
        v = get_path(record, path)
        if v is not None:
            return v
    if key is not None:
        v = find_first_key(record, key)
        if v is not None:
            return v
    return default

def midpoint(obj):
    if obj is None:
        return np.nan
    if isinstance(obj, (int, float, np.integer, np.floating)):
        return float(obj)
    if not isinstance(obj, dict):
        return np.nan
    lo = obj.get("from", obj.get("lower", obj.get("min")))
    hi = obj.get("to", obj.get("upper", obj.get("max")))
    try:
        if lo is None and hi is None:
            return np.nan
        if lo is None:
            return float(hi)
        if hi is None:
            return float(lo)
        return (float(lo) + float(hi)) / 2
    except Exception:
        return np.nan

def bound(obj, side):
    if not isinstance(obj, dict):
        return np.nan
    names = ["from","lower","min"] if side == "from" else ["to","upper","max"]
    for k in names:
        if obj.get(k) is not None:
            try:
                return float(obj[k])
            except Exception:
                return obj[k]
    return np.nan

def extract_records(payload):
    res = payload.get("results")
    if isinstance(res, list):
        return res
    if isinstance(res, dict):
        for k in ("items","content","entries","registerEntries","results"):
            if isinstance(res.get(k), list):
                return res[k]
        if any(k in res for k in ("registerNumber","name","activeLobbyist")):
            return [res]

    found = []
    def walk(x):
        if isinstance(x, dict):
            if "registerNumber" in x and any(
                k in x for k in ("name","activity","activeLobbyist","financialExpensesEuro","registerEntryDetail")
            ):
                found.append(x)
                return
            for v in x.values():
                walk(v)
        elif isinstance(x, list):
            for v in x:
                walk(v)
    walk(payload)

    seen, unique = set(), []
    for x in found:
        marker = (x.get("registerNumber"), x.get("id"), x.get("name"))
        if marker not in seen:
            unique.append(x)
            seen.add(marker)
    return unique

records = extract_records(payload)
print(f"Candidate records: {len(records):,}")

if not records:
    raise RuntimeError("No record-like objects found in the current Bundestag response.")

print("\nKeys in first record:")
print(sorted(records[0].keys()))

print("\nFirst record preview:")
print(json.dumps(records[0], ensure_ascii=False, indent=2)[:5000])

Candidate records: 6,941

Keys in first record:
['accountDetails', 'activitiesAndInterests', 'contracts', 'donators', 'employeesInvolvedInLobbying', 'financialExpenses', 'lobbyistIdentity', 'membershipFees', 'registerEntryDetails', 'registerNumber', 'regulatoryProjects', 'statements']

First record preview:
{
  "registerNumber": "R000774",
  "registerEntryDetails": {
    "registerEntryId": 83965,
    "legislation": "GL2024",
    "detailsPageUrl": "https://www.lobbyregister.bundestag.de/suche/R000774/83965",
    "validFromDate": "2026-08-05T15:58:07.027+02:00"
  },
  "accountDetails": {
    "activeLobbyist": true,
    "firstPublicationDate": "2022-02-21T16:12:29.005+01:00",
    "lastUpdateDate": "2026-08-05T15:58:07.027+02:00",
    "accountHasCodexViolations": false
  },
  "lobbyistIdentity": {
    "identity": "ORGANIZATION",
    "name": "Gesamtverband der Deutschen Versicherungswirtschaft e.V.",
    "entrustedPersonsCount": 93,
    "membersCount": 458,
    "membershipsCount": 21
  },
 

In [4]:
rows, topic_rows = [], []

for rec in records:
    register_number = preferred(
        rec,
        [("registerNumber",), ("registerEntryDetail","account","registerNumber"), ("account","registerNumber")],
        "registerNumber"
    )

    identity = preferred(
        rec,
        [("lobbyistIdentity",), ("registerEntryDetail","lobbyistIdentity")],
        "lobbyistIdentity",
        {}
    ) or {}
    if not isinstance(identity, dict):
        identity = {}

    activity = preferred(
        rec,
        [("activity",), ("registerEntryDetail","activity")],
        "activity",
        {}
    ) or {}
    if not isinstance(activity, dict):
        activity = {"text": activity}

    name = preferred(
        rec,
        [("name",), ("lobbyistIdentity","name"), ("registerEntryDetail","lobbyistIdentity","name")],
        None
    )
    if name is None:
        name = identity.get("name")

    expense_obj = preferred(
        rec,
        [("financialExpensesEuro",), ("registerEntryDetail","financialExpensesEuro")],
        "financialExpensesEuro"
    )
    employee_obj = preferred(
        rec,
        [("employeeCount",), ("registerEntryDetail","employeeCount")],
        "employeeCount"
    )

    fields = preferred(
        rec,
        [("fieldsOfInterest",), ("registerEntryDetail","fieldsOfInterest")],
        "fieldsOfInterest",
        []
    )
    projects = preferred(
        rec,
        [("legislativeProjects",), ("registerEntryDetail","legislativeProjects")],
        "legislativeProjects",
        []
    )
    client_orgs = preferred(
        rec,
        [("clientOrganizations",), ("registerEntryDetail","clientOrganizations")],
        "clientOrganizations",
        []
    )
    client_people = preferred(
        rec,
        [("clientPersons",), ("registerEntryDetail","clientPersons")],
        "clientPersons",
        []
    )

    active = preferred(
        rec,
        [("activeLobbyist",), ("registerEntryDetail","activeLobbyist")],
        "activeLobbyist",
        True
    )

    legal_form = identity.get("legalForm") or {}
    if not isinstance(legal_form, dict):
        legal_form = {}
    address = identity.get("address") or {}
    if not isinstance(address, dict):
        address = {}

    activity_label = (
        activity.get("de") or activity.get("title") or activity.get("text")
        or activity.get("en") or activity.get("code") or "Unknown"
    )

    rows.append({
        "register_number": register_number,
        "name": name or register_number or "Unknown",
        "active_lobbyist": active,
        "activity_code": activity.get("code"),
        "activity_label": activity_label,
        "identity_type": identity.get("identity"),
        "legal_form": (
            legal_form.get("code_de") or legal_form.get("de")
            or legal_form.get("legalFormText") or legal_form.get("title")
            or legal_form.get("code")
        ),
        "city": address.get("city"),
        "expense_midpoint": midpoint(expense_obj),
        "expense_from": bound(expense_obj, "from"),
        "expense_to": bound(expense_obj, "to"),
        "expense_refused": preferred(
            rec,
            [("refuseFinancialExpensesInformation",), ("registerEntryDetail","refuseFinancialExpensesInformation")],
            "refuseFinancialExpensesInformation"
        ),
        "employee_midpoint": midpoint(employee_obj),
        "employee_from": bound(employee_obj, "from"),
        "employee_to": bound(employee_obj, "to"),
        "n_projects": len(as_list(projects)),
        "n_interest_fields": len(as_list(fields)),
        "n_clients": len(as_list(client_orgs)) + len(as_list(client_people)),
        "members": identity.get("members"),
    })

    for f in as_list(fields):
        if not isinstance(f, dict):
            continue
        code_ = f.get("code") or f.get("id")
        label_ = f.get("de") or f.get("title") or f.get("fieldOfInterestText") or f.get("en") or code_
        if code_ or label_:
            topic_rows.append({
                "register_number": register_number,
                "topic_code": str(code_).split("|")[0] if code_ else str(label_),
                "topic": label_,
            })

df = pd.DataFrame(rows).drop_duplicates("register_number", keep="first").reset_index(drop=True)
topics = (
    pd.DataFrame(topic_rows).drop_duplicates()
    if topic_rows
    else pd.DataFrame(columns=["register_number","topic_code","topic"])
)

df["active_lobbyist"] = df["active_lobbyist"].fillna(True)
df["activity_label"] = df["activity_label"].fillna("Unknown")

print(f"Parsed {len(df):,} unique actors")
print(f"Active: {int(df['active_lobbyist'].eq(True).sum()):,}")
print(f"Expenditure bands: {int(df['expense_midpoint'].notna().sum()):,}")
print(f"Employee bands: {int(df['employee_midpoint'].notna().sum()):,}")
print(f"Actors with policy fields: {int(df['n_interest_fields'].gt(0).sum()):,}")
print(f"Actors with legislative projects: {int(df['n_projects'].gt(0).sum()):,}")

display(df.head(10))

✅ Parsed 6,941 unique actors
Active: 6,270
Expenditure bands: 6,820
Employee bands: 0
Actors with policy fields: 6,941
Actors with legislative projects: 0


,register_number,name,active_lobbyist,activity_code,activity_label,identity_type,legal_form,city,expense_midpoint,expense_from,expense_to,expense_refused,employee_midpoint,employee_from,employee_to,n_projects,n_interest_fields,n_clients,members
0,R000774,Gesamtverband der Deutschen Versicherungswirtschaft e.V.,True,ACT_TRADE_ASSOC,Wirtschaftsverband oder Gewerbeverband/-verein,ORGANIZATION,None,None,15835000.5,15830001.0,15840000.0,None,NaN,NaN,NaN,0,37,0,None
1,R001211,Verbraucherzentrale Bundesverband e.V.,True,ACT_NONPROFIT_ORGA_V2,Nichtregierungsorganisation (NGO),ORGANIZATION,None,None,12435000.5,12430001.0,12440000.0,None,NaN,NaN,NaN,0,50,0,None
2,R001243,Verband der Automobilindustrie e.V.,True,ACT_TRADE_ASSOC,Wirtschaftsverband oder Gewerbeverband/-verein,ORGANIZATION,None,None,10275000.5,10270001.0,10280000.0,None,NaN,NaN,NaN,0,60,0,None
3,R000888,BDEW Bundesverband der Energie- und Wasserwirtschaft e.V.,True,ACT_TRADE_ASSOC,Wirtschaftsverband oder Gewerbeverband/-verein,ORGANIZATION,None,None,9885000.5,9880001.0,9890000.0,None,NaN,NaN,NaN,0,37,0,None
4,R000534,Bundesverband der Deutschen Industrie e.V.,True,ACT_TRADE_ASSOC,Wirtschaftsverband oder Gewerbeverband/-verein,ORGANIZATION,None,None,9555000.5,9550001.0,9560000.0,None,NaN,NaN,NaN,0,76,0,None
5,R000476,Verband der Chemischen Industrie e.V.,True,ACT_TRADE_ASSOC,Wirtschaftsverband oder Gewerbeverband/-verein,ORGANIZATION,None,None,9435000.5,9430001.0,9440000.0,None,NaN,NaN,NaN,0,67,0,None
6,R000098,VKU - Verband kommunaler Unternehmen e.V.,True,ACT_TRADE_ASSOC,Wirtschaftsverband oder Gewerbeverband/-verein,ORGANIZATION,None,None,8895000.5,8890001.0,8900000.0,None,NaN,NaN,NaN,0,35,0,None
7,R000726,Campact e.V.,True,ACT_PRIVATE_LAW_ORGA,Privatrechtliche Organisation,ORGANIZATION,None,None,6425000.5,6420001.0,6430000.0,None,NaN,NaN,NaN,0,13,0,None
8,R001795,Wirtschaftsrat der CDU e.V.,True,ACT_PROFESSION_ASSOC,Berufsverband,ORGANIZATION,None,None,6075000.5,6070001.0,6080000.0,None,NaN,NaN,NaN,0,76,0,None
9,R002101,ZVEI e.V.,True,ACT_TRADE_ASSOC,Wirtschaftsverband oder Gewerbeverband/-verein,ORGANIZATION,None,None,5665000.5,5660001.0,5670000.0,None,NaN,NaN,NaN,0,58,0,None


## 3 — Hard data-quality check

If the live feed changes again, the notebook stops here rather than producing empty charts.

In [5]:
availability = pd.DataFrame({
    "Variable": ["Name","Activity","Expenditure","Employees","Policy fields","Legislative projects"],
    "Percent available": [
        100 * df["name"].notna().mean(),
        100 * df["activity_label"].ne("Unknown").mean(),
        100 * df["expense_midpoint"].notna().mean(),
        100 * df["employee_midpoint"].notna().mean(),
        100 * df["n_interest_fields"].gt(0).mean(),
        100 * df["n_projects"].gt(0).mean(),
    ],
})
display(availability.style.format({"Percent available":"{:.1f}%"}))

if (
    df["expense_midpoint"].notna().mean() < 0.01
    and df["employee_midpoint"].notna().mean() < 0.01
):
    raise RuntimeError(
        "Live download succeeded, but resource extraction is still <1%. "
        "The upstream JSON schema changed again. Inspect the first-record preview above. "
        "Downstream analysis is intentionally stopped."
    )

fig = px.bar(
    availability,
    x="Percent available",
    y="Variable",
    orientation="h",
    text=availability["Percent available"].map(lambda x: f"{x:.1f}%"),
    title="Availability of current Lobbyregister variables",
    template=PLOTLY_TEMPLATE,
)
fig.update_xaxes(range=[0,100])
fig.show()

,Variable,Percent available
0,Name,100.0%
1,Activity,100.0%
2,Expenditure,98.3%
3,Employees,0.0%
4,Policy fields,100.0%
5,Legislative projects,0.0%


# 4 — Outputs

In [6]:
active_df = df.loc[df["active_lobbyist"].eq(True)].copy()
spend = active_df.loc[active_df["expense_midpoint"].notna()].copy()

overview = pd.DataFrame({
    "Metric": [
        "Unique actors parsed",
        "Active actors",
        "Active actors with expenditure band",
        "Active actors with employee band",
        "Actors with ≥1 legislative project",
        "Actors with ≥1 policy field",
        "Sum of disclosed expenditure midpoints",
    ],
    "Value": [
        len(df),
        len(active_df),
        len(spend),
        active_df["employee_midpoint"].notna().sum(),
        active_df["n_projects"].gt(0).sum(),
        active_df["n_interest_fields"].gt(0).sum(),
        spend["expense_midpoint"].sum(),
    ],
})
display(overview)

if len(spend):
    top = spend.sort_values("expense_midpoint", ascending=False).head(30)

    display(
        top[[
            "name","register_number","activity_label",
            "expense_midpoint","expense_from","expense_to",
            "employee_midpoint","n_projects","n_interest_fields"
        ]].style.format({
            "expense_midpoint":"€{:,.0f}",
            "expense_from":"€{:,.0f}",
            "expense_to":"€{:,.0f}",
            "employee_midpoint":"{:,.1f}",
        })
    )

    fig = px.bar(
        top.sort_values("expense_midpoint"),
        x="expense_midpoint",
        y="name",
        orientation="h",
        color="activity_label",
        hover_data=["register_number","employee_midpoint","n_projects"],
        title="Highest disclosed annual lobbying expenditure · band midpoint",
        labels={"expense_midpoint":"Annual expenditure midpoint (€)","name":""},
        template=PLOTLY_TEMPLATE,
        height=850,
    )
    fig.show()
else:
    print("No usable expenditure bands extracted.")

,Metric,Value
0,Unique actors parsed,6941.0
1,Active actors,6270.0
2,Active actors with expenditure band,6185.0
3,Active actors with employee band,0.0
4,Actors with ≥1 legislative project,0.0
5,Actors with ≥1 policy field,6270.0
6,Sum of disclosed expenditure midpoints,871327633.5


,name,register_number,activity_label,expense_midpoint,expense_from,expense_to,employee_midpoint,n_projects,n_interest_fields
0,Gesamtverband der Deutschen Versicherungswirtschaft e.V.,R000774,Wirtschaftsverband oder Gewerbeverband/-verein,"€15,835,000","€15,830,001","€15,840,000",nan,0,37
1,Verbraucherzentrale Bundesverband e.V.,R001211,Nichtregierungsorganisation (NGO),"€12,435,000","€12,430,001","€12,440,000",nan,0,50
2,Verband der Automobilindustrie e.V.,R001243,Wirtschaftsverband oder Gewerbeverband/-verein,"€10,275,000","€10,270,001","€10,280,000",nan,0,60
3,BDEW Bundesverband der Energie- und Wasserwirtschaft e.V.,R000888,Wirtschaftsverband oder Gewerbeverband/-verein,"€9,885,000","€9,880,001","€9,890,000",nan,0,37
4,Bundesverband der Deutschen Industrie e.V.,R000534,Wirtschaftsverband oder Gewerbeverband/-verein,"€9,555,000","€9,550,001","€9,560,000",nan,0,76
5,Verband der Chemischen Industrie e.V.,R000476,Wirtschaftsverband oder Gewerbeverband/-verein,"€9,435,000","€9,430,001","€9,440,000",nan,0,67
6,VKU - Verband kommunaler Unternehmen e.V.,R000098,Wirtschaftsverband oder Gewerbeverband/-verein,"€8,895,000","€8,890,001","€8,900,000",nan,0,35
7,Campact e.V.,R000726,Privatrechtliche Organisation,"€6,425,000","€6,420,001","€6,430,000",nan,0,13
8,Wirtschaftsrat der CDU e.V.,R001795,Berufsverband,"€6,075,000","€6,070,001","€6,080,000",nan,0,76
9,ZVEI e.V.,R002101,Wirtschaftsverband oder Gewerbeverband/-verein,"€5,665,000","€5,660,001","€5,670,000",nan,0,58


In [7]:
def gini(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x >= 0]
    if len(x) == 0 or x.sum() == 0:
        return np.nan
    x = np.sort(x)
    n = len(x)
    cum = np.cumsum(x)
    return (n + 1 - 2 * np.sum(cum) / cum[-1]) / n

def top_share(values, pct):
    x = pd.Series(values).dropna().sort_values(ascending=False)
    if len(x) == 0 or x.sum() <= 0:
        return np.nan
    n = max(1, math.ceil(len(x) * pct))
    return x.iloc[:n].sum() / x.sum()

if len(spend):
    g = gini(spend["expense_midpoint"])
    conc = pd.DataFrame({
        "Group":["Top 1%","Top 5%","Top 10%","Top 25%"],
        "Share":[
            top_share(spend["expense_midpoint"],0.01),
            top_share(spend["expense_midpoint"],0.05),
            top_share(spend["expense_midpoint"],0.10),
            top_share(spend["expense_midpoint"],0.25),
        ]
    })
    print(f"Gini coefficient: {g:.3f}")
    display(conc.style.format({"Share":"{:.1%}"}))

    ordered = spend["expense_midpoint"].sort_values().to_numpy()
    pop = np.arange(1,len(ordered)+1)/len(ordered)
    cum = np.cumsum(ordered)/ordered.sum()

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.r_[0,pop], y=np.r_[0,cum], mode="lines", name="Lobbyregister"))
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines", name="Perfect equality", line=dict(dash="dash")))
    fig.update_layout(
        title=f"Lorenz curve of disclosed lobbying expenditure · Gini={g:.3f}",
        xaxis_title="Cumulative share of actors",
        yaxis_title="Cumulative share of expenditure",
        template=PLOTLY_TEMPLATE,
        height=600,
    )
    fig.show()
else:
    g = np.nan

Gini coefficient: 0.825


,Group,Share
0,Top 1%,28.0%
1,Top 5%,56.1%
2,Top 10%,71.4%
3,Top 25%,90.2%


In [8]:
if len(spend):
    by_activity = (
        spend.groupby("activity_label", dropna=False)
        .agg(
            actors=("register_number","nunique"),
            total_midpoint_spend=("expense_midpoint","sum"),
            median_midpoint_spend=("expense_midpoint","median"),
            median_employees=("employee_midpoint","median"),
            total_projects=("n_projects","sum"),
        )
        .sort_values("total_midpoint_spend", ascending=False)
        .reset_index()
    )

    display(by_activity.style.format({
        "total_midpoint_spend":"€{:,.0f}",
        "median_midpoint_spend":"€{:,.0f}",
        "median_employees":"{:,.1f}",
    }))

    fig = px.bar(
        by_activity,
        x="total_midpoint_spend",
        y="activity_label",
        orientation="h",
        text="actors",
        title="Disclosed lobbying expenditure by official activity category",
        labels={"total_midpoint_spend":"Total midpoint expenditure (€)","activity_label":""},
        template=PLOTLY_TEMPLATE,
        height=max(500, 38*len(by_activity)),
    )
    fig.update_yaxes(categoryorder="total ascending")
    fig.show()

if topics.empty:
    print("No policy-field data were present in the parsed response.")
else:
    tc = (
        topics.groupby(["topic_code","topic"])["register_number"]
        .nunique()
        .reset_index(name="actors")
        .sort_values("actors", ascending=False)
        .head(25)
    )
    display(tc)

    fig = px.bar(
        tc.sort_values("actors"),
        x="actors",
        y="topic",
        orientation="h",
        title="Most frequently declared policy fields",
        labels={"actors":"Registered actors","topic":""},
        template=PLOTLY_TEMPLATE,
        height=800,
    )
    fig.show()

,activity_label,actors,total_midpoint_spend,median_midpoint_spend,median_employees,total_projects
0,Unternehmen,1792,"€322,755,782","€55,000",nan,0
1,Wirtschaftsverband oder Gewerbeverband/-verein,715,"€229,150,340","€55,000",nan,0
2,Privatrechtliche Organisation mit Anerkennung der Gemeinnützigkeit nach Abgabenordnung,1386,"€84,020,585","€5,000",nan,0
3,Privatrechtliche Organisation,434,"€67,190,185","€5,000",nan,0
4,Berufsverband,582,"€56,580,249","€5,000",nan,0
5,Nichtregierungsorganisation (NGO),258,"€38,140,113","€15,000",nan,0
6,"Beratungsunternehmen, selbständige Beraterin oder selbständiger Berater",389,"€30,290,153","€5,000",nan,0
7,"Wissenschaft, Hochschule oder Forschungseinrichtung",100,"€15,180,040","€15,000",nan,0
8,Arbeitgeberverband,67,"€8,725,032","€65,000",nan,0
9,"Plattform, Netzwerk, Interessengemeinschaft, Denkfabrik, Initiative, Aktionsbündnis o. ä.",287,"€5,885,088","€5,000",nan,0


,topic_code,topic,actors
105,FOI_SCIENCE_RESEARCH_TECHNOLOGY,"Wissenschaft, Forschung und Technologie",2204
37,FOI_ENVIRONMENT_SUSTAINABILITY,Nachhaltigkeit und Ressourcenschutz,2201
33,FOI_ENVIRONMENT_CLIMATE,Klimaschutz,2063
45,FOI_EU_LAWS,EU-Gesetzgebung,1898
31,FOI_ENERGY_RENEWABLE,Erneuerbare Energien,1572
30,FOI_ENERGY_OVERALL,Allgemeine Energiepolitik,1482
21,FOI_ECONOMY_INDUSTRIAL,Industriepolitik,1409
24,FOI_ECONOMY_SAM_BUSINESS,Kleine und mittlere Unternehmen,1388
82,FOI_MEDIA_DIGITALIZATION,Digitalisierung,1314
62,FOI_HEALTH_SUPPLY,Gesundheitsversorgung,1279


In [9]:
resource = active_df.dropna(subset=["expense_midpoint","employee_midpoint"]).copy()

if len(resource) >= 10:
    resource["plot_size"] = np.sqrt(resource["n_projects"].clip(lower=0) + 1)

    fig = px.scatter(
        resource,
        x="expense_midpoint",
        y="employee_midpoint",
        color="activity_label",
        size="plot_size",
        hover_name="name",
        hover_data={
            "register_number":True,
            "expense_midpoint":":,.0f",
            "employee_midpoint":":.1f",
            "n_projects":True,
            "n_interest_fields":True,
            "plot_size":False,
        },
        log_x=True,
        title="Lobbying resource landscape · budget vs employees · point size=projects",
        labels={
            "expense_midpoint":"Annual expenditure midpoint (€; log scale)",
            "employee_midpoint":"Lobbying employees (band midpoint)",
        },
        template=PLOTLY_TEMPLATE,
        height=750,
    )
    fig.update_traces(marker=dict(opacity=.68))
    fig.show()
else:
    print(f"Resource scatter skipped: only {len(resource)} complete resource records.")

Resource scatter skipped: only 0 complete resource records.


## 5 — Optional data-driven profiles

Unlike the old notebook, clustering cannot crash the workflow. It only runs when enough complete resource records exist.

In [10]:
FEATURES = ["expense_midpoint","employee_midpoint","n_projects","n_interest_fields","n_clients"]
cluster_df = active_df.dropna(subset=["expense_midpoint","employee_midpoint"]).copy()

if len(cluster_df) < 80:
    best_k = None
    best_silhouette = np.nan
    print(f"Clustering skipped: only {len(cluster_df)} complete resource records.")
else:
    feature_cols = []
    for col in FEATURES:
        new = "log_" + col
        cluster_df[new] = np.log1p(pd.to_numeric(cluster_df[col], errors="coerce").clip(lower=0))
        feature_cols.append(new)

    X = StandardScaler().fit_transform(cluster_df[feature_cols])

    candidates = []
    for k in range(2, min(8, max(3, len(cluster_df)//25)) + 1):
        model = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = model.fit_predict(X)
        candidates.append((k, silhouette_score(X, labels), labels))

    best_k, best_silhouette, labels = max(candidates, key=lambda x:x[1])
    cluster_df["profile"] = [f"Profile {x+1}" for x in labels]

    summary = (
        cluster_df.groupby("profile")
        .agg(
            n=("register_number","size"),
            median_expense=("expense_midpoint","median"),
            median_employees=("employee_midpoint","median"),
            median_clients=("n_clients","median"),
            median_projects=("n_projects","median"),
            median_fields=("n_interest_fields","median"),
        )
        .sort_values("median_expense", ascending=False)
    )

    print(f"Selected K={best_k}; silhouette={best_silhouette:.3f}")
    display(summary.style.format({
        "median_expense":"€{:,.0f}",
        "median_employees":"{:,.1f}",
        "median_clients":"{:,.1f}",
        "median_projects":"{:,.1f}",
        "median_fields":"{:,.1f}",
    }))

    pca = PCA(n_components=2, random_state=42)
    xy = pca.fit_transform(X)
    cluster_df["PC1"], cluster_df["PC2"] = xy[:,0], xy[:,1]

    fig = px.scatter(
        cluster_df,
        x="PC1",
        y="PC2",
        color="profile",
        size=np.log1p(cluster_df["expense_midpoint"]) + 1,
        hover_name="name",
        hover_data=["activity_label","register_number","expense_midpoint","employee_midpoint","n_projects"],
        title=f"Empirical lobbying profiles · PCA variance explained={100*pca.explained_variance_ratio_.sum():.1f}%",
        template=PLOTLY_TEMPLATE,
        height=720,
    )
    fig.show()

Clustering skipped: only 0 complete resource records.


In [11]:
n_total = len(df)
n_active = len(active_df)
n_spend = len(spend)
retrieved = payload.get("searchDate") or datetime.now(timezone.utc).isoformat()

if len(spend):
    top10_text = f"{100*top_share(spend['expense_midpoint'],0.10):.1f}%"
    g_text = f"{g:.3f}"
    highest = spend.sort_values("expense_midpoint", ascending=False).iloc[0]
    top_actor_text = f"{highest['name']} (~€{highest['expense_midpoint']:,.0f})"
else:
    top10_text = "unavailable"
    g_text = "unavailable"
    top_actor_text = "unavailable"

if not topics.empty:
    tr = topics.groupby("topic")["register_number"].nunique().sort_values(ascending=False)
    top_topic_text = f"{tr.index[0]} ({int(tr.iloc[0]):,} actors)"
else:
    top_topic_text = "unavailable"

if best_k is None:
    cluster_text = "Clustering was skipped because the complete disclosed-resource subset was too small."
elif best_silhouette < .25:
    cluster_text = f"K={best_k}, silhouette={best_silhouette:.3f}; separation is weak, so a continuum interpretation is preferable."
else:
    cluster_text = f"K={best_k}, silhouette={best_silhouette:.3f}; recurring multivariate profiles are visible, but remain descriptive."

summary_text = (
    f"## Live Lobbyregister summary\n\n"
    f"**Retrieval:** {retrieved}\n\n"
    f"- **{n_total:,}** unique actors parsed.\n"
    f"- **{n_active:,}** actors marked active.\n"
    f"- **{n_spend:,}** active actors with a usable disclosed expenditure band.\n"
    f"- The **top 10%** of disclosed spenders account for **{top10_text}** of midpoint-estimated expenditure.\n"
    f"- **Gini coefficient:** {g_text}.\n"
    f"- Highest disclosed midpoint in this snapshot: **{top_actor_text}**.\n"
    f"- Most frequently parsed policy field: **{top_topic_text}**.\n"
    f"- {cluster_text}\n\n"
    f"### Interpretation limits\n\n"
    f"These are registered and self-declared lobbying data, not a complete measure of political influence. "
    f"Expense ranges are represented by their midpoint. Missing/refused financial information remains missing. "
    f"The analyses are cross-sectional and descriptive."
)

display(Markdown(summary_text))

## Live Lobbyregister summary

**Retrieval:** 2026-08-21T22:17:40.646788+00:00

- **6,941** unique actors parsed.
- **6,270** actors marked active.
- **6,185** active actors with a usable disclosed expenditure band.
- The **top 10%** of disclosed spenders account for **71.4%** of midpoint-estimated expenditure.
- **Gini coefficient:** 0.825.
- Highest disclosed midpoint in this snapshot: **Gesamtverband der Deutschen Versicherungswirtschaft e.V. (~€15,835,000)**.
- Most frequently parsed policy field: **Wissenschaft, Forschung und Technologie (2,204 actors)**.
- Clustering was skipped because the complete disclosed-resource subset was too small.

### Interpretation limits

These are registered and self-declared lobbying data, not a complete measure of political influence. Expense ranges are represented by their midpoint. Missing/refused financial information remains missing. The analyses are cross-sectional and descriptive.

In [13]:
EXPORT = False  #@param {type:"boolean"}
SAVE_RAW = False  #@param {type:"boolean"}

if EXPORT:
    df.to_csv("bundestag_lobbyregister_live_clean.csv", index=False)
    print("Saved: bundestag_lobbyregister_live_clean.csv")
    if not topics.empty:
        topics.to_csv("bundestag_lobbyregister_live_topics.csv", index=False)
        print("Saved: bundestag_lobbyregister_live_topics.csv")

if SAVE_RAW:
    Path("bundestag_lobbyregister_live_raw.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Saved: bundestag_lobbyregister_live_raw.json")

if not EXPORT and not SAVE_RAW:
    print("Set EXPORT=True and/or SAVE_RAW=True if you want files from this run.")

Saved: bundestag_lobbyregister_live_clean.csv
Saved: bundestag_lobbyregister_live_topics.csv


---

### Reproducibility note

Re-running later may change counts, expenditure totals, rankings, policy-field frequencies and clusters because the Bundestag register is updated continuously. The notebook intentionally analyzes the current public snapshot while exposing the source URL, retrieval metadata, missingness and schema preview.